In [0]:
%run ./config

In [0]:
blinkit_schema= StructType(fields=[
    StructField('item_id', StringType()),
    StructField('item_name', StringType()),
    StructField('manufacturer_id', IntegerType()),
    StructField('manufacturer_name', StringType()),
    StructField('city_id', IntegerType()),
    StructField('city_name', StringType()),
    StructField('category', StringType()),
    StructField('date', DateType()),
    StructField('qty_sold', FloatType()),
    StructField('mrp', FloatType())
])

In [0]:
df=spark.read.schema(blinkit_schema).option('header', True).csv(bronze+'/blinkit.csv')

In [0]:
df=df.withColumn('total_revenue', round(col('qty_sold')*col('mrp'),2))\
     .withColumn('data_source', lit('Blinkit'))\
     .withColumnRenamed('qty_sold', 'total_units')\
     .withColumn('sku_name', lower(trim("item_name")))\
     .withColumnRenamed('item_id', 'sku_id')\
     .withColumnRenamed('mrp', 'price')

In [0]:
df=df.select('date','sku_name',"sku_id", "total_units","price", "total_revenue", "data_source")
df=df.na.drop()

In [0]:
df.write.option('header', True).mode('overwrite').csv(silver+'/blinkit.csv')

In [0]:
df=spark.read.csv(silver+'/blinkit.csv', header=True, inferSchema=True)
df=df.select('sku_name').distinct()
display(df)